# Scorecard integration after from-scratch construction

After Chapters 25–30 write manual and automatic bins, WOE/IV and penalised IRLS visibly, assemble their promoted implementations into score scaling, grades, reason codes, and characteristic reports without an external scorecard package.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
from tempfile import TemporaryDirectory

from creditriskbook.data.datasets import load_dataset
from creditriskbook.models import evaluate_pd, split_dataset
from creditriskbook.scorecard import (
    BinningProcess, LogisticScorecard, export_characteristic_report,
    manual_categorical_spec, manual_numeric_spec,
)

bundle = load_dataset("synthetic_retail", n_rows=5_000, seed=202)
train, test = split_dataset(bundle, bundle.frame)
features = ["income", "employment_years", "debt_to_income", "utilisation", "enquiries_6m", "loan_amount", "product", "home_ownership"]
manual = {
    "enquiries_6m": manual_numeric_spec("enquiries_6m", [0, 1, 3, 6]),
    "product": manual_categorical_spec("product", [["personal_loan"], ["credit_card"], ["bnpl"]]),
}

In [ ]:
scorecard = LogisticScorecard(
    binning=BinningProcess(
        numeric_method="monotonic", max_bins=6, prebins=20,
        min_bin_fraction=0.04, min_events=5, manual_specs=manual,
    ),
    l2=1e-3,
).fit(train[features], train[bundle.target])

predicted_pd = scorecard.predict_proba(test[features])[:, 1]
scores = scorecard.score(test[features])
metrics = evaluate_pd(test[bundle.target], predicted_pd)
assert scorecard.model_.converged_
assert scores.min() >= 300 and scores.max() <= 900
print(metrics)
print(scorecard.encoder_.information_values)

In [ ]:
points = scorecard.points_table()
reasons = scorecard.reason_codes(test[features].iloc[:5], top_n=4)
components = scorecard.score_components(test[features].iloc[:5])
print(points.head(12).to_string(index=False))
print(reasons)
print(components[["score", "pd", "rating"]])

with TemporaryDirectory() as directory:
    paths = export_characteristic_report(scorecard, directory)
    assert all(path.exists() for path in paths.values())

## Governance questions

Document every manual cut point, compare monotonic and ChiMerge alternatives, inspect zero-event bins, challenge IV spikes for leakage, and reconcile the row score to the points table.